# RAG Pipeline — Gemini Q&A over Vertex AI Documentation

Builds a full Retrieval-Augmented Generation pipeline that answers questions about
Google Cloud / Vertex AI using the official documentation as its knowledge base.

**The parallel:** This is the same architecture built on AWS Bedrock at City Electric
Supply — S3 corpus → Bedrock Knowledge Bases (embeddings + vector store) → Claude
generation → agentic tool integrations → org-wide rollout. This notebook demonstrates
the equivalent on GCP: Cloud Storage corpus → `gemini-embedding-001` vectors → cosine
retrieval → `gemini-2.5-flash` generation. The concepts transfer directly; the SDK
surfaces differ.

---

**Backend toggle** — set `BACKEND` once in Section 1, used throughout:

| Value | Description | Requirements |
|---|---|---|
| `"ollama"` | Local Ollama server — no API key, no quota | `ollama` running + `gemma4:e2b` pulled |
| `"gemini_api"` | Free Gemini API — fast, 20 req/day limit | `GEMINI_API_KEY` in `.env` |
| `"vertex_ai"` | Enterprise Vertex AI — production path | ADC configured + `GCP_PROJECT_ID` in `.env` |

**Sections:**
1. Setup & Auth
2. Corpus Ingestion
3. Chunking
4. Embedding
5. Retrieval
6. Generation
7. Interactive Q&A ← demo centerpiece
8. RAGAS Evaluation
9. Results Analysis

## Section 1 — Setup & Auth

Set `BACKEND` to one of three values before running the notebook. Every subsequent
section reads from `client` and `embed_client` — no other cells need to change.

The `CostTracker` is active on all backends. On `ollama` and `gemini_api` paths costs
are labeled *estimated at Vertex AI rates* so you can project production spend during
development.

In [41]:
import os
import sys
import time
import json
import subprocess
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

# ── Backend toggle ─────────────────────────────────────────────────────────────
#   "ollama"     — local Ollama server, no API key needed
#                  requires: ollama running + gemma4:e2b pulled
#                  start:    ollama serve  (then: ollama pull gemma4:e2b)
#
#   "gemini_api" — free Gemini API, 20 generation requests/day per project
#                  requires: GEMINI_API_KEY in .env
#
#   "vertex_ai"  — enterprise Vertex AI SDK (Day 4 verification)
#                  requires: gcloud auth application-default login
#                            GCP_PROJECT_ID in .env
BACKEND = "vertex_ai"
# ───────────────────────────────────────────────────────────────────────────────

load_dotenv(dotenv_path="../.env")
sys.path.insert(0, str(Path("../src").resolve()))

GENERATION_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL  = "gemini-embedding-001"

if BACKEND == "ollama":
    from generation import OLLAMA_MODEL, OLLAMA_BASE_URL
    client = None   # generation.py handles Ollama directly when client is None
    GENERATION_MODEL = OLLAMA_MODEL
    print(f"Backend          : Ollama (local, quota-free)")
    print(f"Generation model : {GENERATION_MODEL}")
    print(f"Ollama URL       : {OLLAMA_BASE_URL}")
    print()
    print("Note: embeddings still use Gemini API (gemini-embedding-001).")
    print("      Set GEMINI_API_KEY in .env — only needed for corpus embedding,")
    print("      which is cached after the first run.")
    api_key = os.environ.get("GEMINI_API_KEY")
    if api_key:
        from google import genai
        embed_client = genai.Client(api_key=api_key)
        print(f"      Embedding key: {api_key[:8]}...")
    else:
        embed_client = None
        print("      WARNING: No GEMINI_API_KEY — embedding will fail if cache is missing.")

elif BACKEND == "gemini_api":
    from google import genai

    api_key = os.environ.get("GEMINI_API_KEY")
    assert api_key, "GEMINI_API_KEY not set — check .env"
    client = genai.Client(api_key=api_key)
    embed_client = client
    print(f"Backend          : Gemini API (free tier)")
    print(f"API key          : {api_key[:8]}...")

elif BACKEND == "vertex_ai":
    from google import genai

    project  = os.environ.get("GCP_PROJECT_ID")
    location = os.environ.get("GCP_LOCATION", "us-central1")
    assert project, "GCP_PROJECT_ID not set — check .env"
    # google-genai supports Vertex AI natively — same SDK, same API surface
    # No google-cloud-aiplatform import needed; ADC credentials are picked up automatically
    client       = genai.Client(vertexai=True, project=project, location=location)
    embed_client = client
    print(f"Backend          : Vertex AI (google-genai SDK)")
    print(f"Project          : {project}  Location: {location}")

else:
    raise ValueError(f"Unknown BACKEND: {BACKEND!r}. Use 'ollama', 'gemini_api', or 'vertex_ai'.")

print(f"Generation model : {GENERATION_MODEL}")
print(f"Embedding model  : {EMBEDDING_MODEL}")

from cost import CostTracker
cost_tracker = CostTracker(backend=BACKEND)
print(f"Cost tracker     : active ({BACKEND} rates)")

Backend          : Vertex AI (google-genai SDK)
Project          : gen-lang-client-0948275824  Location: us-south1
Generation model : gemini-2.5-flash
Embedding model  : gemini-embedding-001
Cost tracker     : active (vertex_ai rates)


## Section 2 — Corpus Ingestion

Fetches and cleans GCP/Vertex AI documentation pages into `corpus/`.
The script (`src/build_corpus.py`) is idempotent — safe to re-run, skips nothing
already saved. Expect 15–30 seconds to fetch all pages with polite crawl delays.

Skip this cell if `corpus/manifest.json` already exists from a previous run.

In [43]:
REPO_ROOT = Path.cwd().parent  # Jupyter sets cwd to notebooks/
CORPUS_DIR = REPO_ROOT / "corpus"
MANIFEST_PATH = CORPUS_DIR / "manifest.json"

if MANIFEST_PATH.exists():
    print("Corpus already built — loading manifest.")
else:
    print("Building corpus (this might take a minute) ...")
    result = subprocess.run(
        [sys.executable, str(REPO_ROOT / "src" / "build_corpus.py")],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("build_corpus.py failed")

manifest = json.loads(MANIFEST_PATH.read_text())
pages = manifest["pages"]
skipped = manifest.get("skipped", [])

print(f"\nCorpus: {len(pages)} pages loaded, {len(skipped)} skipped")
for p in pages:
    print(f"  {p['slug']:<40} {p['chars']:>8,} chars")

Corpus already built — loading manifest.

Corpus: 19 pages loaded, 0 skipped
  vertex_ai_overview                          6,718 chars
  gemini_enterprise_platform                 15,949 chars
  gemini_models                              12,829 chars
  gemini_api_overview                        31,630 chars
  gemini_multimodal                           9,453 chars
  embeddings_overview                        16,598 chars
  embeddings_api_reference                   22,403 chars
  rag_overview                                5,947 chars
  rag_quickstart                              7,481 chars
  grounding_overview                         25,230 chars
  context_cache                               5,978 chars
  agent_builder_overview                     38,550 chars
  vertex_ai_search                           11,629 chars
  agent_builder_intro                         4,906 chars
  vector_search_overview                     16,888 chars
  gcp_auth_adc                                5,653 c

## Section 3 — Chunking

Split each corpus document into overlapping word-based chunks.
Each chunk carries: `text`, `source_url`, `chunk_index`, `doc_title`.

- **Chunk size:** 500 words  
- **Overlap:** 50 words (so adjacent chunks share context at boundaries)  
- **Min size:** 30 words (tail fragments below this are discarded)

In [44]:
from chunker import chunk_corpus
from collections import Counter

chunks = chunk_corpus(CORPUS_DIR, manifest)

print(f"Total chunks: {len(chunks)}")
print()

counts = Counter(c["doc_title"] for c in chunks)
for title, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {title:<45} {n:>4} chunks")

print()
print("--- Sample chunk ---")
sample = chunks[10]
print(f"doc_title  : {sample['doc_title']}")
print(f"source_url : {sample['source_url']}")
print(f"chunk_index: {sample['chunk_index']}")
print(f"word count : {len(sample['text'].split())}")
print(f"text       : {sample['text'][:300]}...")

Total chunks: 104

  Gemini API Overview                             13 chunks
  Agent Builder Overview                          13 chunks
  Grounding Overview                               9 chunks
  Embeddings API Reference                         8 chunks
  Vertex AI Access Control                         8 chunks
  Gemini Enterprise Platform                       6 chunks
  Embeddings Overview                              6 chunks
  Vector Search Overview                           6 chunks
  Gemini Models                                    5 chunks
  Vertex AI Search                                 5 chunks
  Gemini Multimodal                                4 chunks
  Gemini Pricing                                   4 chunks
  Vertex AI Overview                               3 chunks
  RAG Quickstart                                   3 chunks
  Context Cache                                    3 chunks
  RAG Overview                                     2 chunks
  Agent Builder Intro

## Section 4 — Embedding

Embed all corpus chunks with `gemini-embedding-001` (3072 dimensions) and cache to disk.
Uses `task_type=RETRIEVAL_DOCUMENT` for corpus chunks — the API optimises the vector
for asymmetric retrieval (document vs. query), which improves recall.

Re-runs are instant: if `corpus/embeddings.npy` already exists the cached file is loaded.

**Rate limit note:** The free-tier Gemini API enforces a requests-per-minute quota on
the embedding endpoint that is much tighter than the generation endpoint. In testing,
batches of 50 hit 429 `RESOURCE_EXHAUSTED` reliably; batches of 10 with a 2-second
inter-batch pause succeed most of the time. The embedder includes exponential backoff
(10 s base, up to 5 retries) to recover from the occasional 429 that still slips
through. On the Vertex AI path (Day 4) this is not a concern — enterprise quotas are
provisioned per project.

In [45]:
from embedder import Embedder, load_embeddings
from cost import _emb_cost

EMBEDDINGS_PATH = CORPUS_DIR / "embeddings.npy"
CHUNKS_PATH     = CORPUS_DIR / "chunks.json"

if EMBEDDINGS_PATH.exists() and CHUNKS_PATH.exists():
    print("Cached embeddings found — loading from disk.")
    embeddings, chunks = load_embeddings(CORPUS_DIR)
else:
    embedder = Embedder(embed_client)
    embeddings = embedder.embed_and_save(chunks, CORPUS_DIR)
    _, chunks = load_embeddings(CORPUS_DIR)  # reload to confirm round-trip

    # One-time corpus embedding cost (only shown when embeddings are first computed)
    total_chars = sum(len(c["text"]) for c in chunks)
    corpus_cost = _emb_cost(total_chars)
    print(f"\n  One-time corpus embedding cost (estimated at Vertex AI rates):")
    print(f"    {total_chars:,} chars across {len(chunks)} chunks → ${corpus_cost:.6f}")

print(f"\nembeddings shape : {embeddings.shape}")
print(f"chunks count     : {len(chunks)}")
print(f"vector norm[0]   : {float((embeddings[0]**2).sum()**0.5):.4f}")
print()
print("--- Sample chunk + vector head ---")
s = chunks[10]
print(f"doc_title  : {s['doc_title']}")
print(f"chunk_index: {s['chunk_index']}")
print(f"vector[:6] : {embeddings[10][:6].tolist()}")

Cached embeddings found — loading from disk.

embeddings shape : (104, 3072)
chunks count     : 104
vector norm[0]   : 1.0000

--- Sample chunk + vector head ---
doc_title  : Gemini Models
chunk_index: 1
vector[:6] : [-0.020244726911187172, 0.0014868582366034389, 0.00956457294523716, -0.065340556204319, -0.006471665110439062, -0.002483459422364831]


## Section 5 — Retrieval

Given a query, embed it with `task_type=RETRIEVAL_QUERY` and rank all corpus chunks
by cosine similarity. Because `gemini-embedding-001` returns unit vectors, cosine
similarity reduces to a single matrix–vector dot product — no normalisation step needed.

Returns the top-k chunks with their similarity score, source URL, and doc title.

In [46]:
from retrieval import retrieve

TEST_QUERY = "How do I get text embeddings with the Gemini API?"
results = retrieve(TEST_QUERY, client, embeddings, chunks, k=5)

print(f"Query: {TEST_QUERY}\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] Score: {r['score']:.4f} | {r['doc_title']}")
    print(f"     Source: {r['source_url']}")
    print(f"     {r['text'][:200]}...")
    print()

Query: How do I get text embeddings with the Gemini API?

[1] Score: 0.8001 | Embeddings Overview
     Source: https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings
     for. Limit: five texts of up to 2,048 tokens per text for all models except textembedding-gecko@001 . The max input token length for textembedding-gecko@001 is 3072. For gemini-embedding-001 , each re...

[2] Score: 0.7999 | Embeddings API Reference
     Source: https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/text-embeddings-api
     five texts of up to 2,048 tokens per text for all models except textembedding-gecko@001 . The max input token length for textembedding-gecko@001 is 3072. For gemini-embedding-001 , each request can on...

[3] Score: 0.7887 | Embeddings API Reference
     Source: https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/text-embeddings-api
     Home Documentation AI and ML Vertex AI Generative AI on Vertex AI API reference Tex

## Section 6 — Generation

Passes the retrieved chunks to Gemini as a numbered context block and instructs it to
answer **only from that context**. The system prompt explicitly forbids speculation —
if the answer isn't in the retrieved chunks, the model says so.

Source URLs are de-duplicated and surfaced alongside the answer so every claim is
traceable back to the original documentation page.

In [47]:
from generation import generate

TEST_QUERY = "What embedding models does Vertex AI support and what dimensions do they produce?"
context_chunks = retrieve(TEST_QUERY, client, embeddings, chunks, k=5)
answer = generate(TEST_QUERY, context_chunks, client)

print(f"Query: {TEST_QUERY}\n")
print(answer["text"])
print("\nSources:")
for url in answer["sources"]:
    print(f"  {url}")

Query: What embedding models does Vertex AI support and what dimensions do they produce?

Vertex AI supports the following embedding models and their output dimensions:

*   **gemini-embedding-001**: up to 3072 dimensions
*   **text-embedding-005**: up to 768 dimensions
*   **text-multilingual-embedding-002**: up to 768 dimensions
*   **multilingual-e5-small**: Up to 384 dimensions
*   **multilingual-e5-large**: Up to 1024 dimensions

Sources:
  https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings
  https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/text-embeddings-api
  https://cloud.google.com/vertex-ai/docs/vector-search/overview


## Section 7 — Interactive Q&A

The demo centerpiece. `ask(question)` runs the full RAG pipeline and prints:

1. **Retrieved chunks** — top-5 by cosine similarity, with scores and source URLs
2. **Generated answer** — grounded in retrieved context only
3. **Sources** — deduplicated list of documentation pages cited

Change `QUESTION` in the cell below to ask anything about Vertex AI / Gemini.

In [48]:
def ask(question: str, k: int = 5) -> None:
    sep = "─" * 72
    cost_tracker.reset_exchange()

    print(sep)
    print(f"  QUESTION: {question}")
    print(sep)

    # Retrieve
    results = retrieve(question, client, embeddings, chunks, k=k)
    query_chars = len(question)
    cost_tracker.record_embedding(query_chars)

    print(f"\n  RETRIEVED CHUNKS  (top {k} by cosine similarity)\n")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] score={r['score']:.4f}  {r['doc_title']}")
        print(f"       {r['source_url']}")
        print(f"       {r['text'][:160].strip()}...")
        print()

    # Generate
    answer = generate(question, results, client)
    usage  = answer.get("usage", {})
    if usage:
        cost_tracker.record_generation(
            usage.get("input_tokens", 0),
            usage.get("output_tokens", 0),
            usage.get("model", "gemini-2.5-flash"),
        )

    print(f"  ANSWER\n")
    for line in answer["text"].strip().splitlines():
        print(f"  {line}")
    print(f"\n  SOURCES")
    for url in answer["sources"]:
        print(f"    • {url}")
    print()
    print(cost_tracker.exchange_summary())
    print(sep)


# ── Change this question and re-run the cell ──────────────────────────────────
QUESTION = "How does Vertex AI RAG Engine work, and what retrieval backends does it support?"
ask(QUESTION)

────────────────────────────────────────────────────────────────────────
  QUESTION: How does Vertex AI RAG Engine work, and what retrieval backends does it support?
────────────────────────────────────────────────────────────────────────

  RETRIEVED CHUNKS  (top 5 by cosine similarity)

  [1] score=0.8097  RAG Overview
       https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-overview
       Home Documentation AI and ML Vertex AI Generative AI on Vertex AI Guides Vertex AI RAG Engine overview The VPC-SC security controls and CMEK are supported by Ve...

  [2] score=0.8023  RAG Overview
       https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-overview
       component in Vertex AI RAG Engine searches through its knowledge base to find information that is relevant to the query. Generation : The retrieved information...

  [3] score=0.7666  RAG Quickstart
       https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-quickstart
       Ho

## Section 8 — RAGAS Evaluation

Runs four RAG quality metrics over `eval/test_set.json` (12 questions, 8 topics)
using Gemini as an LLM judge.

| Metric | What it measures |
|---|---|
| **Faithfulness** | Are all answer claims supported by the retrieved context? |
| **Answer Relevancy** | Does the answer actually address the question asked? |
| **Context Precision** | Are the retrieved chunks relevant to the question? |
| **Context Recall** | Were all chunks needed to answer the question retrieved? |

**Judge LLM fallback chain:** Vertex AI `gemini-2.5-flash` → free-tier
`gemini-2.5-flash-lite` → local Ollama. The script auto-selects based on what's
available — set `GCP_PROJECT_ID` in `.env` to prefer the Vertex AI path.

**Runtime:** ~3 minutes on Vertex AI with `max_workers=4`.

In [49]:
import subprocess, sys
from pathlib import Path

# Streams output line-by-line so progress is visible while running (~3 min on Vertex AI)
proc = subprocess.Popen(
    [sys.executable, str(REPO_ROOT / "eval" / "evaluate.py")],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    print(f"\nEvaluation exited with code {proc.returncode}")

Loading corpus embeddings...
  104 chunks, 3072 dims
Loading test set...
  12 questions
  Pipeline backend : Vertex AI (us-south1)

Generating answers for each question...
  [1/12] What is the output dimensionality of gemini-embedding-001 and how does...
  [2/12] What are the six steps in the Vertex AI RAG Engine pipeline?...
  [3/12] How does Grounding with Google Search work in Vertex AI, and what is t...
  [4/12] What is the search order that Application Default Credentials uses to ...
  [5/12] What is context caching in Vertex AI and what are the two types availa...
  [6/12] What technology underpins Vertex AI Vector Search and what Google prod...
  [7/12] What are the responsible AI limitations that Vertex AI documentation i...
  [8/12] What is the minimum token count required for explicit context caching ...
  [9/12] What task types should be used when embedding corpus documents versus ...
  [10/12] What are the per-request limits for the Vertex AI Text Embeddings API?...
  [11/1

## Section 9 — Results Analysis

Loads the saved `eval/results.json` and surfaces patterns in the scores.
The goal isn't just to show passing numbers — it's to show *where the pipeline
could be improved*, which is the more credible FDE answer.

In [50]:
import json
from pathlib import Path

results_path = REPO_ROOT / "eval" / "results.json"
if not results_path.exists():
    print("No results.json found — run Section 8 first.")
else:
    data = json.loads(results_path.read_text())
    means = data["means"]
    per_q  = data["per_question"]

    print(f"Judge LLM: {data['judge_llm']}")
    print()

    # Aggregate scores
    print("  Aggregate scores:\n")
    for metric, val in means.items():
        if val is not None:
            bar  = "█" * int(val * 10)
            gate = "✓" if val >= 0.7 else "✗"
            print(f"  {gate}  {metric:<22} {val:.3f}  {bar}")
        else:
            print(f"  ?  {metric:<22}   N/A")

    # Highlight weak spots (Context Precision < 0.7)
    weak = [q for q in per_q if (q.get("Context Precision") or 1.0) < 0.7]
    if weak:
        print(f"\n  Context Precision weak spots (< 0.70):\n")
        for q in weak:
            cp = q.get("Context Precision")
            s  = f"{cp:.3f}" if cp is not None else " N/A"
            print(f"    [{s}]  {q['question'][:70]}...")
        print()
        print("  Interpretation: retrieved chunks are not all relevant to the question.")
        print("  Fix: tighten corpus quality for these topics, or reduce retrieval k.")

Judge LLM: Vertex AI gemini-2.5-flash (us-south1)

  Aggregate scores:

  ✓  Faithfulness           0.984  █████████
  ✓  Answer Relevancy       0.861  ████████
  ✓  Context Precision      0.712  ███████
  ✓  Context Recall         0.938  █████████

  Context Precision weak spots (< 0.70):

    [0.679]  What is the output dimensionality of gemini-embedding-001 and how does...
    [0.500]  What are the six steps in the Vertex AI RAG Engine pipeline?...
    [0.333]  What are the per-request limits for the Vertex AI Text Embeddings API?...
    [0.200]  How does Vertex AI RAG Engine support VPC Service Controls and what se...

  Interpretation: retrieved chunks are not all relevant to the question.
  Fix: tighten corpus quality for these topics, or reduce retrieval k.
